![DB Academy](../Includes/images/common/db-academy.png)


<div style="
  border-left: 4px solid #7b1fa2;
  background: #f3e5f5;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#4a148c; margin-bottom:6px; font-size: 1.1em;">Lab Information</strong>
  <div style="color:#333;">

This is a comprehensive demonstration of adding an ML model to a DAB. Due to live class time constraints, this content is optional and best explored at the end of class.
  </div>
</div>



# 07L Bonus - Adding ML to Engineering Workflows with Declarative Automation Bundles (DABs)

### Estimated Duration: 25-30 minutes

## Overview

Your data engineering workflow is in good shape. The ML team has now asked you to add a model-inference task so the workflow runs predictions against a registered Unity Catalog model the team already trained. **You don't need to know any ML for this lab.** Your goal is to wire the existing model into the bundle: declare the right variables, add a new task that calls the inference notebook, and promote the same bundle through `development` and `stage` targets.

## Learning Objectives

By the end of this lab, you will be able to:

1. **Add new bundle variables** (including a `lookup` variable for `cluster_id`) to a pre-existing `variables.yml`.
2. **Extend an existing job YAML** with a new task that depends on prior tasks and passes parameters into a notebook.
3. **Use `databricks bundle summary`** to inspect what will be deployed before deploying it.
4. **Validate, deploy, run, and destroy** the bundle against `development`, then promote the same bundle to `stage`.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select All-Purpose Compute</strong>
  <div style="color:#333;">

This notebook requires **all-purpose compute** (Dedicated). Serverless is not supported for this notebook.

Follow these steps to attach an all-purpose compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your `labuser_USERNAME` cluster.
    - By default, the notebook might use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

⚠️ **NOTE:** If the cluster shows a **terminated** state (red dot in the cluster picker), it needs to be started before you can attach. Click the cluster, then **Start**, and wait a few minutes until you see a green dot.
  </div>
</div>


## REQUIRED - DATA SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Data Setup</strong>
  <div style="color:#333;">

Recall that your environment was set up using the **0 - REQUIRED - Course Setup and Authentication** notebook.

If you end your lab or your lab session times out, your environment will be reset. You will need to rerun the **0 - REQUIRED - Course Setup and Authentication** notebook to recreate the catalogs and refresh your Databricks CLI credentials.

  </div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this course.

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of them. It dynamically references the information needed to run the course.

**NOTE:** This will take 2-3 minutes to set up and create the models.

In [0]:
%run ../Includes/Classroom-Setup-7L

✅ Vocareum workspace detected.
✅ Using existing Vocareum catalog: 'labuser15884059_1784281908'.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.2/28.2 MB 125.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.2/614.2 kB 44.5 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-0e3d30f3-a51c-4996-87f4-92cdb5a2d2f3
    Can't uninstall 'blinker'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.0.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-0e3d30f3-a51c-4996-87f4-92cdb5a2d2f3
    Can't uninstall 'mlflow-skinny'. No files were found to uninstall.
Note: you may need to rest

Checking to see if model exists in labuser15884059_1784281908_1_dev...
No versions found in labuser15884059_1784281908_1_dev_1_dev; you can train and register a new model.
No existing model found; training & registering diabetes_model in labuser15884059_1784281908_1_dev


2026/07/17 10:04:35 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
2026/07/17 10:04:51 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be time-consuming. To speed up, explicitly specify the conda_env or pip_requirements when calling log_model().
2026/07/17 10:05:37 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/2116215346137463/1442d9a5f33348f2a70b99400b0bdc85/artifacts/model/sparkml, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
Successfully registered model 'labuser15884059_1784281908_1_dev.default.diabetes_model_dev'.
🔗 Created versio

Model registered successfully for environment: labuser15884059_1784281908_1_dev
No existing model found; training & registering diabetes_model in labuser15884059_1784281908_2_stage


...

DATABRICKS_HOST set to:  https://dbc-7f5587c4-8fc8.cloud.databricks.com
DATABRICKS_TOKEN set.


Installed Databricks CLI v0.298.0 at /root/bin/databricks.


Catalog check for the labs passed.


Information,Value
"DEV catalog variable: ""catalog_dev"":",labuser15884059_1784281908_1_dev
"STAGE catalog variable: ""catalog_stage"":",labuser15884059_1784281908_2_stage
"PROD catalog variable: ""catalog_prod"":",labuser15884059_1784281908_3_prod


Compute,Status,Details
All-Purpose,✓ Match,Version 17.3


## B. Lab Scenario

Congratulations! You've successfully built the bulk of your workflow. 

The ML team has asked you to ensure your tests meet their requirements for inferencing a model they've deployed in the dev environment. **You don't need to learn ML for this lab.** Just attach the model to the workflow using the bundle you've already built.

**Optional task before starting:** if you have ML knowledge, you can inspect the pre-trained model by navigating to **Experiments**. Otherwise, your goal is simply to add it to your bundle.

## C. Pre-flight Checks

Confirm the Databricks CLI is authenticated against your workspace before starting the lab tasks. Run the cells below and check for errors.

In [0]:
%sh 
databricks catalogs list

Name                    Type                  Comment
dbacademy               MANAGED_CATALOG       
dbacademy_cdc_diabetes  DELTASHARING_CATALOG  # CDC Diabetes Health Indicators

## Attribution

This course uses the CDC Diabetes Health Indicators Dataset, which is licensed under CC0: Public Domain.

## License Information

You can find more details about the "CDC Diabetes Health Indicators Dataset" on [Kaggle](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset) and[UC Irvine ML Repository](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)

The "CDC Diabetes Health Indicators Dataset" is distributed under the CC0: Public Domain license. Please review the license terms before using the dataset.

For any questions or inquiries regarding the dataset or its usage, please refer to the above pages for contact information.

---
Note: This README is provided as part of the dataset attribution for educational purposes. Please ensure compliance w

## D. Task 1 - Update `variables.yml`

In the folder where this notebook lives, you'll find a sub-folder named **TODO - Lab DABs Workflow**. 

You'll edit a couple of files there to attach the registered ML model to the workflow. 

**You do not need to know what the model does**, your goal is to understand how to attach an additional Unity Catalog asset (a registered ML model in this case).

#### What's in the bundle

1. Navigate to the **src/** folder. You'll find:
    - **dlt_pipelines/**
    - **helpers/**,
    - Two notebooks: **Final Visualization** and **Inference**. 
        - The notebook this lab focuses on is **Inference**.

2. In the **Inference** notebook, look at the section **Parameterize the notebook for our workflow and passing variables**. Two variables are read by the notebook:
    - **base_model_name**: the registered model name
    - **silver_table_name**: the silver table name and location, expected as **catalog.schema.silver_sample_ml**

3. In a separate tab, open **resources/variables.yml**. You'll add a few variables here.

### Step 1.1 - Add `base_model_name` to **variables.yml**

Add a `base_model_name` variable in the section marked 

- To find the default value, locate the model in your dev catalog (**labuser_UNIQUE_ID_1_dev.default**) under **Models**.

### Step 1.2 - View the `silver_table_name` to **variables.yml**

View the `silver_table_name` variable. 
  - The default value should be set to `${var.username}_1_dev.default.silver_sample_ml`.

### Step 1.3 - Add `cluster_id` to **variables.yml**

The inference task needs an existing cluster. Define a `cluster_id` variable. You have four options:

- **Option 1:** Define a `lookup` variable on `username` and reference it via `${var.username}`.
- **Option 2:** Use `lookup` and set the `cluster` value to `${workspace.current_user.userName}`.
- **Option 3:** Hardcode the default value using the `lookup` method.
- **Option 4:** Find your cluster ID by navigating to **Compute** in the left menu, opening your cluster, clicking the kebab menu, and choosing **View JSON**. Copy the cluster ID near the top of the JSON. Alternatively, run `print(spark.conf.get("spark.databricks.clusterUsageTags.clusterId"))` in a new cell. 
  - Paste this value as the `default` for `cluster_id`.



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Summary
  </strong>
  <div style="color:#333;">

After this task, **variables.yml** should have three new variables: `base_model_name`, `silver_table_name`, and `cluster_id`. Each has a description and a default value.

**HINT:** Variable substitution and lookups documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/bundles/variables) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/bundles/variables) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/bundles/variables)

  </div>
</div>



## E. Task 2 - Update `resources/dabs_workflow_with_ml.job.yml`

Now that **variables.yml** is updated, extend the workflow with a new inference task. (You will not be configuring the Spark Declarative Pipeline in this step.)

Navigate to **resources/job/** and open **dabs_workflow_with_ml.job.yml**. You'll see all the existing tasks. 

Add a new task with the following constraints under the comment `### Complete your ML TASK HERE`:

1. Add task name (`task_key`) **ML_test**.

2. The task must depend on **Health_ETL** via `depends_on`.

3. Add an `existing_cluster_id` key whose value references the `cluster_id` variable you created in Task 1.
    - Reference your `cluster_id` variable: `${var.cluster_id}`. 

4. Add a `notebook_task` containing `notebook_path`, `base_parameters`, and `source`:

    - `notebook_path` should reference the **Inference** notebook (**HINT**: Go back to folders).

    - `base_parameters` should have **3** keys: 
        - Two referencing the new variables (`base_model_name` and `silver_table_name`)
        - One that references the dev catalog. 
        - **HINT:** use a variable that's already pre-configured in **variables.yml**.

    - You can also add a description if you'd like.

**HINT:** Use the existing tasks in this file as templates.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Summary
  </strong>
  <div style="color:#333;">

After this task, **dabs_workflow_with_ml.job.yml** has a new task wired to the existing **Health_ETL** dependency, and you're ready to validate the bundle.

  </div>
</div>




## F. Task 3 - View the Bundle Summary

Use `databricks bundle summary` to print out the resources defined in the project and the names that will be generated after deploying the bundle.

**NOTE:** Each `%sh` cell starts a fresh shell, so you must `cd` into the **TODO - Lab DABs Workflow** folder *and* run the CLI command in the **same** cell.

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle summary

Name: ml_health_etl_bundle
Target: development
Workspace:
  User: labuser15884059_1784281908@vocareum.com
  Path: /Workspace/Users/labuser15884059_1784281908@vocareum.com/.bundle/ml_health_etl_bundle/development
Resources:
  Jobs:
    ml_health_etl_workflow:
      Name: [dev labuser15884059_1784281908] ml_health_etl_workflow_development
      URL:  (not deployed)
  Pipelines:
    health_etl_pipeline:
      Name: [dev labuser15884059_1784281908] health_etl_pipeline_with_ml_development
      URL:  (not deployed)


<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
     Troubleshooting
  </strong>
  <div style="color:#333;">
If you see the following error after validating your bundle, the format of your notebook could be incorrect.

`Error: notebook xxx not found`. 

Check the format of your notebook and adjust accordingly. 

  </div>
</div>



## G. Task 4 - Validate the Bundle

Validate your **databricks.yml** bundle configuration file using the Databricks CLI for the `development` target. Confirm validation succeeds. If there is an error, fix the YAML and re-run.

**HINT:** `databricks bundle` CLI commands documentation:
[AWS](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/cli/bundle-commands) |
[GCP](https://docs.databricks.com/gcp/en/dev-tools/cli/bundle-commands)

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle validate

Name: ml_health_etl_bundle
Target: development
Workspace:
  User: labuser15884059_1784281908@vocareum.com
  Path: /Workspace/Users/labuser15884059_1784281908@vocareum.com/.bundle/ml_health_etl_bundle/development

Validation OK!


##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
cd "./TODO - Lab DABs Workflow"
pwd
databricks bundle validate -t development
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

</details>

## H. Task 5 - Deploy to the `development` Target

Deploy the bundle to the `development` target.

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle deploy -t development

Source-linked deployment is enabled. Deployed resources reference the source files in your working tree instead of separate copies.
Deploying resources...
Updating deployment state...
Deployment complete!


##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle deploy -t development
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

</details>

Navigate to **Jobs & Pipelines** and open the Job `[dev labuser_UNIQUE_ID] ml_health_etl_workflow_development`



#### Checkpoint - Dev Deployment
![Ml Job Deploy](../Includes/images/ml-lab/dev-deployment-job-checkpoint.png)


## I. Task 6 - Run the `development` Workflow

Run the deployed workflow against the `development` target. 

The job key in the bundle is `ml_health_etl_workflow`.

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle run ml_health_etl_workflow -t development

Run URL: https://dbc-7f5587c4-8fc8.cloud.databricks.com/?o=7474652962058183#job/916854824143172/run/62106213161675

2026-07-17 10:12:10 "[dev labuser15884059_1784281908] ml_health_etl_workflow_development" RUNNING
2026-07-17 10:15:07 "[dev labuser15884059_1784281908] ml_health_etl_workflow_development" TERMINATED SUCCESS


Output:
Task Unit_Tests:

Task Visualization:

Task ML_test:



#### Checkpoint - Dev Run 
![Ml Job Deploy](../Includes/images/ml-lab/dev-job-run.png)


<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
     Troubleshooting
  </strong>
  <div style="color:#333;">
If you see the following error after running your bundle, the format of your notebook could be incorrect.

```
Error: Task Health_ETL failed!
Error:
Please refer to the logs for this pipeline in the pipelines page.
```

Check the format of your notebook for the SDP and adjust accordingly!

  </div>
</div>



##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle run ml_health_etl_workflow -t development
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

</details>

## J. Task 7 - Destroy the `development` Bundle

Clean up the `development` deployment.

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle destroy -t development --auto-approve

The following resources will be deleted:
  delete resources.jobs.ml_health_etl_workflow
  delete resources.pipelines.health_etl_pipeline

This action will result in the deletion of the following Lakeflow Spark Declarative Pipelines along with the
Streaming Tables (STs) and Materialized Views (MVs) managed by them:
  delete resources.pipelines.health_etl_pipeline

All files and directories at the following location will be deleted: /Workspace/Users/labuser15884059_1784281908@vocareum.com/.bundle/ml_health_etl_bundle/development

Deleting files...
Destroy complete!


## K. Task 8 - Promote the Bundle to `stage`

Imagine you've reviewed your code, analyzed coverage, and so on, and you're ready to deploy and test in a staging environment. DABs make this easy: you change one CLI flag (`-t stage`) and the bundle's `stage` target overrides take care of the rest.

Walk through the same lifecycle, this time against the `stage` target. First, take a moment to look at the `stage` block inside **databricks.yml** and notice what overrides have already been set for you.

### Step 8.1 - Bundle summary for `stage`

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle summary -t stage

Name: ml_health_etl_bundle
Target: stage
Workspace:
  User: labuser15884059_1784281908@vocareum.com
  Path: /Workspace/Users/labuser15884059_1784281908@vocareum.com/.bundle/ml_health_etl_bundle/stage
Resources:
  Jobs:
    ml_health_etl_workflow:
      Name: [dev labuser15884059_1784281908] ml_health_etl_workflow_stage
      URL:  (not deployed)
  Pipelines:
    health_etl_pipeline:
      Name: [dev labuser15884059_1784281908] health_etl_pipeline_with_ml_stage
      URL:  (not deployed)


### Step 8.2 - Validate `stage`

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle validate -t stage

Name: ml_health_etl_bundle
Target: stage
Workspace:
  User: labuser15884059_1784281908@vocareum.com
  Path: /Workspace/Users/labuser15884059_1784281908@vocareum.com/.bundle/ml_health_etl_bundle/stage

Validation OK!


### Step 8.3 - Deploy to `stage`

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle deploy -t stage

Source-linked deployment is enabled. Deployed resources reference the source files in your working tree instead of separate copies.
Deploying resources...
Updating deployment state...
Deployment complete!


### Step 8.4 - Run the `stage` workflow

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle run ml_health_etl_workflow -t stage

Run URL: https://dbc-7f5587c4-8fc8.cloud.databricks.com/?o=7474652962058183#job/1030648212584384/run/993094621756026

2026-07-17 10:15:54 "[dev labuser15884059_1784281908] ml_health_etl_workflow_stage" RUNNING
2026-07-17 10:18:37 "[dev labuser15884059_1784281908] ml_health_etl_workflow_stage" TERMINATED SUCCESS


Output:
Task Unit_Tests:

Task ML_test:

Task Visualization:



#### Checkpoint - Stage Run

![Ml Job Deploy Stage](../Includes/images/ml-lab/stage-job-run.png)

### Step 8.5 - Destroy the `stage` bundle

In [0]:
%sh
cd "./TODO - Lab DABs Workflow"
databricks bundle destroy -t stage --auto-approve

The following resources will be deleted:
  delete resources.jobs.ml_health_etl_workflow
  delete resources.pipelines.health_etl_pipeline

This action will result in the deletion of the following Lakeflow Spark Declarative Pipelines along with the
Streaming Tables (STs) and Materialized Views (MVs) managed by them:
  delete resources.pipelines.health_etl_pipeline

All files and directories at the following location will be deleted: /Workspace/Users/labuser15884059_1784281908@vocareum.com/.bundle/ml_health_etl_bundle/stage

Deleting files...
Destroy complete!


## Conclusion

Nice work. In this lab you wired a registered Unity Catalog ML model into an existing engineering bundle without changing the rest of the workflow:

1. Added the `base_model_name`, `silver_table_name`, and `cluster_id` variables to **variables.yml**.
2. Added a new inference task to **dabs_workflow_with_ml.job.yml**, depending on **Health_ETL**, calling the **Inference** notebook with three `base_parameters`.
3. Used `databricks bundle summary` to inspect the resolved bundle before deploying.
4. Validated, deployed, ran, and destroyed the bundle against `development`.
5. Promoted the same bundle to `stage` with a single `-t stage` flag and ran the same lifecycle there.

## Next Steps

Try building your own DAB from scratch using what you learned here. It helps to grow the workflow incrementally, one task at a time, validating after each change so problems stay easy to isolate.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>